# Importing Libraries and Preliminary Setup

In [ ]:
# Check available conda environments - senpi should be listed
!conda info --envs

# check that current executable should report as python.exe in senpi directory
import sys
sys.executable

In [ ]:
# Preliminary operation forcing Jupyter to reload all modules during execution
%load_ext autoreload
%autoreload 2

# fix step if reading hdf5 file leads to error: Can't read data (Can't open directory)
# %pip uninstall h5py
# %pip install h5py

In [ ]:
# Main Libraries
import torch
from torch import nn
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import timeit
from timeit import default_timer as timer
import importlib
import pathlib
from pathlib import Path
from typing import List
from typing import Dict
from typing import Union
import math

import kornia
from kornia.filters import gaussian_blur2d, bilateral_blur
from enum import Enum
import PIL
from PIL import Image
from PIL.PngImagePlugin import PngInfo
import cv2
from tqdm import tqdm
import traceback
import h5py
# import hdf5plugin
# import tables
from dv import AedatFile
from dv import LegacyAedatFile
import yacs

In [ ]:
# In case dependency issue, should be: ('1.26.2', '2.1.4', '10.0.1')
np.__version__, pd.__version__, PIL.__version__

In [ ]:
# Check if GPU available and report pytorch version. Should be: 2.12 or higher
torch.cuda.is_available(), torch.__version__

In [ ]:
# Created Libraries
# import senpi
from senpi import *

# data_io.basic_utils
# import data_io.basic_utils
from senpi.data_io.basic_utils import *

# data_manip
# import data_manip.algs
# import data_manip.conversions
# import data_manip.filters
# import data_manip.preprocessing
from senpi.data_manip.algs import *
from senpi.data_manip.conversions import *
from senpi.data_manip.filters import *
from senpi.data_manip.preprocessing import *
from senpi.data_manip.computation import *

from senpi.data_vis.visualization import *

# data_gen.reconstruction
# import data_gen.reconstruction
from senpi.data_gen.reconstruction import *

# constants
from senpi import constants

# Loading a Large Dataset for Later Use in Practical Testing

In [ ]:
# Load events from a DAVIS event camera (.aedat)
from dv import LegacyAedatFile
file_path = "./senpi_test_datasets/night_drive_data/night_drive.aedat"

start_time = timer()

with LegacyAedatFile(file_path) as f:
	events = []
	first_event = next(f)
	start_timestamp = first_event.timestamp
	events.append([0, constants.WIDTH_DAVIS - 1 - first_event.x, constants.HEIGHT_DAVIS - 1 - first_event.y, 1 if first_event.polarity else -1])
	for event in f:
		cur_e_T = event.timestamp - start_timestamp
		if (cur_e_T > 10000000): # fetching no more data past 10 seconds of time
			break
		events.append([cur_e_T, constants.WIDTH_DAVIS - 1 - event.x, constants.HEIGHT_DAVIS - 1 - event.y, 1 if event.polarity else -1])

end_time = timer()
print(f"Load Time: {round(end_time - start_time, 2)} seconds for {len(events)} events.")

In [ ]:
# events in form [t, x, y, p]
len(events), events[:10]

In [ ]:
# Demo to convert events in dataframe
night_drive_dataframe = pd.DataFrame(events, columns=constants.DAVIS_DATA_ORDER)
print(night_drive_dataframe.shape)
night_drive_dataframe.head(10)

In [ ]:
# Demo as numpy array
night_drive_arr = night_drive_dataframe.to_records(index=False).view(np.ndarray)
print(night_drive_arr.shape)
night_drive_arr[:10]

In [ ]:
# as pytorch tensor
night_drive_tens = torch.tensor(events)
print(night_drive_tens.shape)
night_drive_tens[:10]

## Ensuring Loaded Data is Identical Across All Argument Types

In [ ]:
# show enforce consistency - transfering between contains does not corrupt data

# Convert existing dataframe to torch
night_drive_df_tensor = torch.tensor(night_drive_dataframe.to_numpy())

# Convert existing struct array to torch
t_nd = night_drive_arr['t']
x_nd = night_drive_arr['x']
y_nd = night_drive_arr['y']
p_nd = night_drive_arr['p']
night_drive_arr_tensor = torch.tensor(np.stack((t_nd, x_nd, y_nd, p_nd), axis=1))

# test consistency
print(f'Number of incorrect elements in dataframe -> torch conversion: {torch.sum(night_drive_df_tensor != night_drive_tens)} / {len(events)}')
print(f'Number of incorrect elements in struct array -> torch conversion:{torch.sum(night_drive_arr_tensor != night_drive_tens)} / {len(events)}')
# print(torch.any((night_drive_df_tensor != night_drive_tens)).item(), torch.any((night_drive_arr_tensor != night_drive_tens)).item())

# SENPI Testing

In [ ]:
# load paths to appropriate test files of different extensions
HDF5_FILE = "./senpi_test_datasets/spinner_data/spinner.hdf5"
RAW_FILE = "./senpi_test_datasets/spinner_data/spinner.raw"
AEDAT_FILE = "./senpi_test_datasets/night_drive_data/night_drive.aedat"
LOAD_FILE = "./senpi_test_datasets/sample_data_1/sample_data_1.csv"
SAVE_FILE = "./senpi_test_datasets/sample_data_1/sample_data_1_{obj_type}.csv"

# determined desired order to organize loaded data + determine size of event batches
PROGRAM_INIT_ORDER = ['t', 'x', 'y', 'p']
PROGRAM_ORDER = ['b', 't', 'x', 'y', 'p']
BATCH_SIZE = 128
# SAVE_ORDER = ['b', 'x', 'y', 'p', 't']

## Basic Utils

### Loading to Pandas DataFrame

#### CSV File

In [ ]:
loaded_df = load_to_df(LOAD_FILE, delim=',', order=constants.PROPHESEE_DATA_ORDER, ret_order=PROGRAM_INIT_ORDER, dtypes=constants.PROPHESEE_PD_NP_DTYPES, \
                       transform_polarity=False, sub_first_timestamp=False)
print(loaded_df.shape)
loaded_df[:10]

#### RAW File

In [ ]:
loaded_raw_df = load_to_df(RAW_FILE, delim=',', order=constants.PROPHESEE_DATA_ORDER, ret_order=PROGRAM_INIT_ORDER,
                           dtypes=constants.PROPHESEE_PD_NP_DTYPES, transform_polarity=False, sub_first_timestamp=True)
print(loaded_raw_df.shape)
print(loaded_raw_df[:10])

### Loading to NumPy Structured Array

#### CSV File

In [ ]:
loaded_arr = load_to_np_arr(LOAD_FILE, order=constants.PROPHESEE_DATA_ORDER, ret_order=PROGRAM_INIT_ORDER, dtypes=constants.PROPHESEE_PD_NP_DTYPES, \
                            transform_polarity=True, sub_first_timestamp=False)
print(loaded_arr.shape)
loaded_arr[:10]

#### AEDAT File

In [ ]:
loaded_aedat_arr = load_to_np_arr(AEDAT_FILE, ret_order=PROGRAM_INIT_ORDER, dtypes=constants.PROPHESEE_PD_NP_DTYPES, \
                                   transform_polarity=True, sub_first_timestamp=True)
print(loaded_aedat_arr.shape)
loaded_aedat_arr[:10]

### Loading to Torch Tensor

In [ ]:
device = "cpu"
loaded_tens = load_to_tensor(LOAD_FILE, delim=',', order=constants.PROPHESEE_DATA_ORDER, ret_order=PROGRAM_INIT_ORDER, \
                             dtypes=constants.PROPHESEE_TORCH_DTYPES, device=device, sub_first_timestamp=False)
loaded_tens.shape, loaded_tens[:10]

#### HDF5

In [ ]:
# device_hdf5 = "cpu"
# loaded_tens_hdf5 = load_to_tensor(HDF5_FILE, ret_order=PROGRAM_INIT_ORDER, dtypes=constants.PROPHESEE_TORCH_DTYPES, device=device_hdf5)
# loaded_tens_hdf5.shape, loaded_tens_hdf5[:10]

#### Comparing Data Objects Loaded from RAW and HDF5 Files Corresponding to the Same Data

In [ ]:
# loaded_raw_df[-10:], loaded_tens_hdf5[-10:]

### Saving DataFrame to CSV

In [ ]:
save_df_to_csv(loaded_df, f=SAVE_FILE.format(obj_type="PD"), delim=',', save_order=constants.PROPHESEE_DATA_ORDER, revert_polarity=True)

In [ ]:
loaded_df_2 = load_to_df(SAVE_FILE.format(obj_type="PD"), delim=',', order=constants.PROPHESEE_DATA_ORDER, dtypes=constants.PROPHESEE_PD_NP_DTYPES, transform_polarity=False)
loaded_df_2

### Saving NumPy Structured Array to CSV

In [ ]:
save_np_arr_to_csv(loaded_arr, f=SAVE_FILE.format(obj_type="ARR"), delim=",", save_order=constants.PROPHESEE_DATA_ORDER, revert_polarity=True, with_index=True)

In [ ]:
file_columns = ['i']
file_columns.extend(constants.PROPHESEE_DATA_ORDER) # extend returns None
print(file_columns)
loaded_df_2 = load_to_df(SAVE_FILE.format(obj_type="ARR"), delim=',', order=file_columns, dtypes=constants.PROPHESEE_PD_NP_DTYPES, transform_polarity=False)
loaded_df_2

### Saving Tensor to CSV

In [ ]:
save_tensor_to_csv(loaded_tens, f=SAVE_FILE.format(obj_type="TENS"), delim=" ", revert_polarity=False)

In [ ]:
loaded_df_2 = load_to_df(SAVE_FILE.format(obj_type="TENS"), delim=' ', order=PROGRAM_INIT_ORDER, dtypes=constants.PROPHESEE_PD_NP_DTYPES, transform_polarity=False)
loaded_df_2

## Data Manipulations

### Algorithms

#### FlipX

In [ ]:
flip_x_alg = FlipXAlg(constants.WIDTH_DAVIS - 1)

##### Pandas DataFrame

In [ ]:
flip_x_df = night_drive_dataframe.copy()
print(flip_x_df.shape)
flip_x_df.head(10)

In [ ]:
flip_x_alg.transform(flip_x_df)
print(flip_x_df.shape)
flip_x_df.head(10)

In [ ]:
flip_x_alg.transform(flip_x_df, modify_original=False)[:10], flip_x_df[:10]

##### Structured NumPy Array

In [ ]:
flip_x_arr = night_drive_arr.copy()
flip_x_arr[:10], flip_x_arr.shape

In [ ]:
flip_x_alg.transform(flip_x_arr)
print(flip_x_arr.shape)
flip_x_arr[:10]

In [ ]:
flip_x_alg.transform(flip_x_arr, modify_original=False)[:10], flip_x_arr[:10]

##### Torch Tensor

In [ ]:
flip_x_tensor = night_drive_tens.detach().clone()
flip_x_tensor[:10], flip_x_tensor.shape

In [ ]:
flip_x_alg.transform(flip_x_tensor, order=PROGRAM_INIT_ORDER)
flip_x_tensor[:10]

In [ ]:
flip_x_alg.transform(flip_x_tensor, order=PROGRAM_INIT_ORDER, modify_original=False)[:10], flip_x_tensor[:10]

#### FlipY

In [ ]:
flip_y_alg = FlipYAlg(constants.HEIGHT_DAVIS - 1)

##### Pandas DataFrame

In [ ]:
flip_y_df = night_drive_dataframe.copy()
print(flip_y_df.shape)
flip_y_df.head(10)

In [ ]:
flip_y_alg.transform(flip_y_df)
print(flip_y_df.shape)
flip_y_df.head(10)

In [ ]:
flip_y_alg.transform(flip_y_df, modify_original=False)[:10], flip_y_df[:10]

##### Structured NumPy Array

In [ ]:
flip_y_arr = night_drive_arr.copy()
flip_y_arr[:10], flip_y_arr.shape

In [ ]:
flip_y_alg.transform(flip_y_arr)
print(flip_y_arr.shape)
flip_y_arr[:10]

In [ ]:
flip_y_alg.transform(flip_y_arr, modify_original=False)[:10], flip_y_arr[:10]

##### Torch Tensor

In [ ]:
flip_y_tensor = night_drive_tens.detach().clone()
flip_y_tensor[:10], flip_y_tensor.shape

In [ ]:
flip_y_alg.transform(flip_y_tensor, order=PROGRAM_INIT_ORDER)
flip_y_tensor[:10]

In [ ]:
flip_y_alg.transform(flip_y_tensor, order=PROGRAM_INIT_ORDER, modify_original=False)[:10], flip_y_tensor[:10]

#### Invert Polarity

In [ ]:
invert_polarity_alg = InvertPolarityAlg()

##### Pandas DataFrame

In [ ]:
invert_polarity_df = night_drive_dataframe.copy()
print(invert_polarity_df.shape)
invert_polarity_df.head(10)

In [ ]:
invert_polarity_alg.transform(invert_polarity_df)
print(invert_polarity_df.shape)
invert_polarity_df.head(10)

In [ ]:
invert_polarity_alg.transform(invert_polarity_df, modify_original=False).head(10), invert_polarity_df.head(10)

In [ ]:
invert_polarity_df_2 = loaded_df.copy()
print(invert_polarity_df_2.shape)
invert_polarity_df_2.head(10)

In [ ]:
invert_polarity_alg.transform(invert_polarity_df_2, binary_polarity_mode=True, modify_original=False).head(10), invert_polarity_df_2.head(10)

##### Structured NumPy Array

In [ ]:
invert_polarity_arr = night_drive_arr.copy()
invert_polarity_arr[:10], invert_polarity_arr.shape

In [ ]:
invert_polarity_alg.transform(invert_polarity_arr)
print(invert_polarity_arr.shape)
invert_polarity_arr[:10]

In [ ]:
invert_polarity_alg.transform(invert_polarity_arr, modify_original=False)[:10], invert_polarity_arr[:10]

##### Torch Tensor

In [ ]:
invert_polarity_tensor = night_drive_tens.detach().clone()
invert_polarity_tensor[:10], invert_polarity_tensor.shape

In [ ]:
invert_polarity_alg.transform(invert_polarity_tensor, order=PROGRAM_INIT_ORDER)
invert_polarity_tensor[:10]

In [ ]:
invert_polarity_alg.transform(invert_polarity_tensor, order=PROGRAM_INIT_ORDER, modify_original=False)[:10], invert_polarity_tensor[:10]

### Conversions

In [ ]:
convert_df = night_drive_dataframe.copy()
print(convert_df.shape, convert_df['t'].dtype)
convert_df.head(10)

In [ ]:
df_microseconds_to_seconds(convert_df)
print(convert_df.shape, convert_df['t'].dtype)
convert_df.head(10)

In [ ]:
df_seconds_to_microseconds(convert_df)
print(convert_df.shape, convert_df['t'].dtype)
convert_df.head(10)

#### Demonstrating Resulting Floating Point Error

In [ ]:
convert_df[(night_drive_dataframe['t'] != convert_df['t'])].index

In [ ]:
convert_df.iloc[266], night_drive_dataframe.iloc[266]

### Filters

#### Polarity Filter

In [ ]:
positive_filter = PolarityFilter(True)
negative_filter = PolarityFilter(False)

##### Pandas DataFrame

In [ ]:
print(night_drive_dataframe.shape, night_drive_dataframe['p'].unique())
night_drive_dataframe.head(7)

In [ ]:
negative_events_dataframe = negative_filter.filter_events(night_drive_dataframe)
print(negative_events_dataframe.shape, negative_events_dataframe['p'].unique())
negative_events_dataframe.head(7)

In [ ]:
print(loaded_df.shape, loaded_df['p'].unique())
loaded_df.head(7)

In [ ]:
positive_events_dataframe = positive_filter.filter_events(loaded_df, binary_polarity_mode=True)
print(positive_events_dataframe.shape, positive_events_dataframe['p'].unique())
positive_events_dataframe.head(7)

##### Structured NumPy Array

In [ ]:
print(night_drive_arr.shape, np.unique(night_drive_arr['p']))
night_drive_arr[:7]

In [ ]:
negative_events_arr = negative_filter.filter_events(night_drive_arr)
print(negative_events_arr.shape, np.unique(negative_events_arr['p']))
negative_events_arr[:7]

##### Torch Tensor

In [ ]:
param_ind = {PROGRAM_INIT_ORDER[ind]: ind for ind in range(len(PROGRAM_INIT_ORDER))}

In [ ]:
night_drive_tens.shape, night_drive_tens[:10], night_drive_tens[:, param_ind['p']].unique()

In [ ]:
positive_events_tensor = positive_filter.filter_events(night_drive_tens, order=PROGRAM_INIT_ORDER)
positive_events_tensor.shape, positive_events_tensor[:10], positive_events_tensor[:, param_ind['p']].unique()

#### Background Activity Filter (BAF)

In [ ]:
ba_filter = BAFilter(time_threshold=1000, height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS)

##### Pandas DataFrame

In [ ]:
start_time = timer()
baf_df = ba_filter.filter_events(night_drive_dataframe)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(baf_df.shape)
baf_df.head(10)

In [ ]:
# To Use for Comparison Later
baf_df_tensor = torch.tensor(baf_df.to_numpy(), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(baf_df[c].dtype) for c in baf_df.columns]))
baf_df_tensor.shape, baf_df_tensor[:10], baf_df_tensor.dtype

##### NumPy Structured Array

In [ ]:
start_time = timer()
baf_arr = ba_filter.filter_events(night_drive_arr)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(baf_arr.shape)
baf_arr[:10]

In [ ]:
# To Use for Comparison Later
t = baf_arr['t']
x = baf_arr['x']
y = baf_arr['y']
p = baf_arr['p']

baf_arr_tensor = torch.tensor(np.stack((t, x, y, p), axis=1), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(baf_arr[c].dtype) for c in baf_arr.dtype.names]))
baf_arr_tensor.shape, baf_arr_tensor[:10]

##### Torch Tensor

In [ ]:
start_time = timer()
baf_tensor = ba_filter.filter_events(night_drive_tens, order=PROGRAM_INIT_ORDER)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(baf_tensor.shape)
baf_tensor[:10]

##### Ensuring Equality of Filter Output for Each Argument Type

In [ ]:
print(torch.any((baf_df_tensor != baf_tensor)).item(), torch.any((baf_arr_tensor != baf_tensor)).item())

#### Inceptive Events Filter (Based on Inceptive Event Time-Surfaces (IETS))

In [ ]:
ie_filter = IEFilter(12000, 12000, height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS)

##### Pandas DataFrame

In [ ]:
start_time = timer()
ie_df = ie_filter.filter_events(night_drive_dataframe)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ie_df.shape)
ie_df.head(10)

In [ ]:
# To Use for Comparison Later
ie_df_tensor = torch.tensor(ie_df.to_numpy(), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(ie_df[c].dtype) for c in ie_df.columns]))
ie_df_tensor.shape, ie_df_tensor[:10], ie_df_tensor.dtype

##### NumPy Structured Array

In [ ]:
start_time = timer()
ie_arr = ie_filter.filter_events(night_drive_arr)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ie_arr.shape)
ie_arr[:10]

In [ ]:
# To Use for Comparison Later
t = ie_arr['t']
x = ie_arr['x']
y = ie_arr['y']
p = ie_arr['p']

ie_arr_tensor = torch.tensor(np.stack((t, x, y, p), axis=1), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(ie_arr[c].dtype) for c in ie_arr.dtype.names]))
ie_arr_tensor.shape, ie_arr_tensor[:10]

##### Torch Tensor

In [ ]:
start_time = timer()
ie_tensor = ie_filter.filter_events(night_drive_tens, order=constants.DAVIS_DATA_ORDER)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ie_tensor.shape)
ie_tensor[:10]

##### Ensuring Equality of Filter Output for Each Argument Type

In [ ]:
print(torch.any((ie_df_tensor != ie_tensor)).item(), torch.any((ie_arr_tensor != ie_tensor)).item())

#### YNoise Filter (Event Density + Optional Hot Pixel Filtering)

In [ ]:
ynoise_filter = YNoiseFilter(time_context_delta=5000, space_window_size=5, density_threshold=3, height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS)

##### Pandas DataFrame

In [ ]:
start_time = timer()
ynoise_df = ynoise_filter.filter_events(night_drive_dataframe)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ynoise_df.shape)
ynoise_df.head(10)

In [ ]:
# To Use for Comparison Later
ynoise_df_tensor = torch.tensor(ynoise_df.to_numpy(), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(ynoise_df[c].dtype) for c in ynoise_df.columns]))
ynoise_df_tensor.shape, ynoise_df_tensor[:10], ynoise_df_tensor.dtype

##### NumPy Structured Array

In [ ]:
start_time = timer()
ynoise_arr = ynoise_filter.filter_events(night_drive_arr)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ynoise_arr.shape)
ynoise_arr[:10]

In [ ]:
# To Use for Comparison Later
t = ynoise_arr['t']
x = ynoise_arr['x']
y = ynoise_arr['y']
p = ynoise_arr['p']

ynoise_arr_tensor = torch.tensor(np.stack((t, x, y, p), axis=1), dtype=get_dominant_torch_dtype([get_torch_dtype_from_np_dtype(ynoise_arr[c].dtype) for c in ynoise_arr.dtype.names]))
ynoise_arr_tensor.shape, ynoise_arr_tensor[:10]

##### Torch Tensor

In [ ]:
start_time = timer()
ynoise_tensor = ynoise_filter.filter_events(night_drive_tens, order=constants.DAVIS_DATA_ORDER)
end_time = timer()
print(f"Time taken: {end_time - start_time} seconds")

print(ynoise_tensor.shape)
ynoise_tensor[:10]

##### Ensuring Equality of Filter Output for Each Argument Type

In [ ]:
print(torch.any((ynoise_df_tensor != ynoise_tensor)).item(), torch.any((ynoise_arr_tensor != ynoise_tensor)).item())

### Preprocessing

#### NumPy Events Structured Array to Torch Tensor

##### `np_events_to_tensor` (1)

In [ ]:
tensor_from_arr = np_events_to_tensor(night_drive_arr, transform_polarity=True, stack_ret_order=PROGRAM_INIT_ORDER) # function of interest
tensor_from_arr[:10], tensor_from_arr.dtype

In [ ]:
print(torch.any((night_drive_tens != night_drive_arr_tensor)).item(), torch.any((tensor_from_arr != night_drive_arr_tensor)).item())
print(torch.any((night_drive_tens != tensor_from_arr)).item())

#### Partition Events NumPy Structured Array into List of NumPy Structured Array Event Batches based on Batch Size

##### `partition_np_list_batch` (7)

In [ ]:
start_time = timer()
arr_list_batches = partition_np_list_batch(night_drive_arr, batch_size=BATCH_SIZE) # function of interest
end_time = timer()
print(f"Time taken to partition NumPy array into batches: {end_time - start_time} seconds")

In [ ]:
print(len(arr_list_batches), arr_list_batches[0].shape, arr_list_batches[-1].shape, len(arr_list_batches[0][0]))
arr_list_batches[3][:10], arr_list_batches[3].shape # 1 batch of 128

#### Add a Batch Column to Torch Tensor Based on Batch Size

##### `add_batch_col` (6)

In [ ]:
night_drive_tensor_copy = night_drive_tens.detach().clone()

In [ ]:
add_batch_col(night_drive_tensor_copy, batch_size=BATCH_SIZE, modify_original=True) # Note: modify_original uses .data attribute to change tensor data directly
print(night_drive_tensor_copy.device, PROGRAM_ORDER)
print(night_drive_tensor_copy[:10], night_drive_tensor_copy[:10].shape, sep="\n")
print(night_drive_tensor_copy[-10:], night_drive_tensor_copy[-10:].shape, sep="\n")

In [ ]:
batched_night_drive_tens = add_batch_col(night_drive_tens, batch_size=BATCH_SIZE, modify_original=False)
print(batched_night_drive_tens.device, PROGRAM_ORDER)
print(batched_night_drive_tens[:10], batched_night_drive_tens[:10].shape, sep="\n")
print(batched_night_drive_tens[-10:], batched_night_drive_tens[-10:].shape, sep="\n")

In [ ]:
print(night_drive_tens.device, PROGRAM_INIT_ORDER)
print(night_drive_tens[:10], night_drive_tens[:10].shape, sep="\n")
print(night_drive_tens[-10:], night_drive_tens[-10:].shape, sep="\n")

#### List of NumPy Structured Arrays (Batched Events) to Torch Tensor (with Batch Column)

##### `list_np_events_to_tensor` (4)

In [ ]:
start_time = timer()
tensor_from_arr_list = list_np_events_to_tensor(arr_list_batches, transform_polarity=False, stack_ret_order=PROGRAM_ORDER)
end_time = timer()
print(f"Time taken to convert structured NumPy array list to tensor: {end_time - start_time} seconds.")

In [ ]:
print(tensor_from_arr_list.device, PROGRAM_ORDER)
print(tensor_from_arr_list[:10], tensor_from_arr_list[-10:], sep="\n")
print(torch.any((tensor_from_arr_list != batched_night_drive_tens)).item())

#### Torch Tensor (with or without Batch Column) to List of NumPy Structured Arrays (Batched Events)

##### `events_tensor_to_np_list` (5)

In [ ]:
# if batch column is not in tensor, must specify this through stack_ret_order parameter; stack_ret_order is important
start_time = timer()
arr_list_from_tensor = events_tensor_to_np_list(batched_night_drive_tens, batch_size=BATCH_SIZE, stack_ret_order=PROGRAM_ORDER)
end_time = timer()
print(f"Time taken to convert tensor to list of NumPy structured arrays: {end_time - start_time} seconds")

In [ ]:
print(len(arr_list_from_tensor), arr_list_from_tensor[0].shape, arr_list_from_tensor[-1].shape, len(arr_list_from_tensor[0][0]))
arr_list_from_tensor[3][:10], arr_list_from_tensor[3].shape # 1 batch of 128

#### Torch Tensor Event Batch to Image Representation

##### `events_tensor_batch_to_img` (2)

In [ ]:
# Only a single batch
# import traceback
# try:
ret_dict = events_tensor_batch_to_img(events=batched_night_drive_tens, batch_size=BATCH_SIZE * 16, \
												 height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS, \
												 time_surface_mode="most_recent", separate_polarity=True,
                                                 displacement_mode=True, window_size=21, \
												 stack_order=PROGRAM_ORDER)
# except Exception:
# 	print(traceback.format_exc())
img_tensor_batch_ND = ret_dict['ret_img_tensor']
full_time_surface = ret_dict['running_time_surface']
# img_tensor_batch_ND = events_tensor_batch_to_img(events=batched_night_drive_tens, batch_size=BATCH_SIZE * 16, \
# 												 height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS, \
# 												 time_surface_mode="exponential_decay_sum", separate_polarity=True, \
#                                                displacement_mode=True, window_size=21, \
# 												 stack_order=PROGRAM_ORDER)

In [ ]:
print(img_tensor_batch_ND.shape, full_time_surface.shape)

In [ ]:
# (img_tensor_batch_ND != 0).nonzero(), len((img_tensor_batch_ND != 0).nonzero())

##### Visualizing Results

In [ ]:
plot_time_surface(img_tensor_batch_ND, plot_type='3d')

In [ ]:
plot_time_surface(full_time_surface, plot_type='3d')

In [ ]:
# plot_time_surface(img_tensor_batch_ND, plot_type='2d')

#### Torch Event Batch to Image Volume Representation

##### `events_tensor_batch_to_vol` (3)

In [ ]:
param_ind = {PROGRAM_ORDER[ind] : ind for ind in range(len(PROGRAM_ORDER))}
NBINS = batched_night_drive_tens[-1, param_ind['b']].item() + 1
param_ind, NBINS

Optimizing Generation of Parameters

In [ ]:
# Lightning fast vectorized NumPy code to achieve the same thing as above
b_column = batched_night_drive_tens[:, param_ind['b']].to(torch.int64)
t_column = batched_night_drive_tens[:, param_ind['t']]

start_times_vect = np.zeros(NBINS, dtype=get_np_dtype_from_torch_dtype(t_column.dtype))
durations_vect = np.zeros(NBINS, dtype=get_np_dtype_from_torch_dtype(t_column.dtype))

sub_batch_nums, sub_batch_start_indices = np.unique(b_column, return_index=True)
start_times_vect[sub_batch_nums] = t_column[sub_batch_start_indices]

sub_batch_end_indices = np.searchsorted(b_column, sub_batch_nums, side='right') - 1 # upper bound
durations_vect[sub_batch_nums] = t_column[sub_batch_end_indices] - t_column[sub_batch_start_indices]

Testing Function

In [ ]:
start_time = timer()

img_vol_from_tensor_batch = events_tensor_batch_to_vol(events=batched_night_drive_tens, total_batch_size=batched_night_drive_tens.size(0), \
													   height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS, \
													   start_times=start_times_vect, durations=durations_vect, nbins=NBINS, stack_order=PROGRAM_ORDER)

end_time = timer()
time_taken = end_time - start_time
print(f"Time to generate image volume from tensor: {time_taken: .5f} seconds.")

In [ ]:
img_vol_from_tensor_batch.shape

In [ ]:
plot_time_surface(img_vol_from_tensor_batch[-1000], plot_type='3d')

In [ ]:
nonzero_entries = (img_vol_from_tensor_batch[0] != 0).nonzero()
print(nonzero_entries.shape)
nonzero_entries[:10]

## Data Manipulation Computation Testing

### Test 1

In [ ]:
ba_filter_elem = BAFilter(time_threshold=1000, height=constants.HEIGHT_DAVIS, width=constants.WIDTH_DAVIS)
ynoise_filter_elem = YNoiseFilter(time_context_delta=5000, space_window_size=5, \
                                                  density_threshold=3, height=constants.HEIGHT_DAVIS, \
                                                    width=constants.WIDTH_DAVIS)

seq_compute_pipeline = SequentialCompute(ba_filter_elem,
                                          ynoise_filter_elem)
# seq_compute_pipeline = SequentialCompute([ba_filter_elem,
#                                           ynoise_filter_elem]) 
seq_compute_pipeline.set_param_dict({'order': PROGRAM_INIT_ORDER})

event_pipeline_tensor = seq_compute_pipeline(night_drive_tens)

In [ ]:
event_pipeline_tensor.shape, event_pipeline_tensor[:10]

In [ ]:
plot_events(event_pipeline_tensor, order=PROGRAM_INIT_ORDER, separate_polarity=True)

### Test 2

In [ ]:
invert_polarity_elem = InvertPolarityAlg()
ynoise_filter_elem_2 = ynoise_filter = YNoiseFilter(time_context_delta=5000, space_window_size=5, \
                                                  density_threshold=10, height=constants.HEIGHT_DAVIS, \
                                                    width=constants.WIDTH_DAVIS)

seq_compute_pipeline_2 = SequentialCompute([invert_polarity_elem,
                                          ynoise_filter_elem_2])
seq_compute_pipeline_2.set_param_dict({'order': PROGRAM_INIT_ORDER,
                                       'modify_original': False})

event_pipeline_tensor_2 = seq_compute_pipeline_2(night_drive_tens)

In [ ]:
event_pipeline_tensor_2.shape, event_pipeline_tensor_2[:10]

In [ ]:
night_drive_tens.shape, night_drive_tens[:10]

In [ ]:
night_drive_tens[night_drive_tens[:, 0] == 2800]

### Test 3

In [ ]:
invert_polarity_elem_2 = InvertPolarityAlg()
flip_x_elem = FlipXAlg(constants.WIDTH_DAVIS - 1)

seq_compute_pipeline_3 = SequentialCompute([invert_polarity_elem_2,
                                          flip_x_elem])
seq_compute_pipeline_3.set_param_dict({'order': PROGRAM_INIT_ORDER,
                                       'modify_original': False,
                                       'binary_polarity_mode': False})

event_pipeline_tensor_3 = seq_compute_pipeline_3(night_drive_tens)

In [ ]:
event_pipeline_tensor_3

In [ ]:
night_drive_tens

## Data Generation

In [ ]:
DATA_FOLDER = "./data/"
FPS = 200

### Reconstruction

In [ ]:
baf_tensor[:10], baf_tensor.shape

In [ ]:
img_generator = ImgFramesFromEventsGenerator(width=constants.WIDTH_DAVIS, height=constants.HEIGHT_DAVIS, cutoff_frequency=2 * math.pi)
start_time = timer()

nd_time_unit = TimeUnit.MICROSECONDS
tensor_ref = event_pipeline_tensor # event_pipeline_tensor_3 # baf_tensor # night_drive_tens
vid_file_name = "BAFYNoiseSeqTest" # "ElemAlgsInvFlipX"

# events_per_frame is not necessary if gen_frames_mode is "fps"
img_generator.generate_frames(tensor_ref, events_per_frame=1500, order=PROGRAM_INIT_ORDER, gen_frames_mode="fps", time_unit=nd_time_unit, enable_gpu=False, \
                              accumulate_frame_list=False, frames_tensor_list_device="cpu", save_images=False, \
                              generate_video=True, fps=FPS, output_parent_dir=DATA_FOLDER, video_file_name=vid_file_name,
                              first_event_start_timestamp=True)

# img_generator.generate_frames(tensor_ref, events_per_frame=1500, order=PROGRAM_INIT_ORDER, gen_frames_mode="fps", time_unit=nd_time_unit, enable_gpu=False, \
#                               accumulate_frame_list=False, frames_tensor_list_device="cpu", save_images=False, \
#                               spatial_smoothing_method="bilateral", kernel_size=1, \
#                               generate_video=True, fps=200, output_parent_dir=DATA_FOLDER, video_file_name=None)

end_time = timer()
time_taken = end_time - start_time
print(f"Time taken to generate frames: {time_taken: .5f} seconds")

## Data Visualization

In [ ]:
event_pipeline_tensor[int(len(event_pipeline_tensor) / 4)]

In [ ]:
denoised_to_plot = event_pipeline_tensor[:int(len(event_pipeline_tensor) / 4)]

In [ ]:
plot_events(events=denoised_to_plot, order=PROGRAM_INIT_ORDER, separate_polarity=False)

In [ ]:
plot_events(events=denoised_to_plot, order=PROGRAM_INIT_ORDER, separate_polarity=True)

In [ ]:
print(len((night_drive_tens[:, 0] < 2726055).nonzero()))
partial_full_data = night_drive_tens[night_drive_tens[:, 0] < 2726055]
partial_full_data.shape, partial_full_data[:10]

plot_events(events=partial_full_data, order=PROGRAM_INIT_ORDER, separate_polarity=True)

# --- END OF CORE TESTING --- 